In [1]:
"""
Pipeline complet de détection de nodules pulmonaires
Seed fixe pour la reproductibilité
"""

import os
import random
import math
import shutil
import yaml

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from scipy.ndimage import gaussian_filter
from ultralytics import YOLO
from torchvision import transforms
from typing import List, Tuple
from tqdm import tqdm

# ─────────────────────────────────────────────
# 0. SEED GLOBALE
# ─────────────────────────────────────────────
SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")

# ─────────────────────────────────────────────
# 1. CHEMINS ET CONFIGURATION
# ─────────────────────────────────────────────
DIR_PATH_LIDC   = "lidc_png_16_bit"               # images LIDC (16-bit, annotées)
DIR_PATH_NIH    = "nih_filtered_images"            # images NIH  (8-bit, classification seulement)
CSV_BBOX        = "localization_labels.csv"        # x, y centre nodule
CSV_LABELS      = "classification_labels.csv"      # file_name, label, LIDC_ID
MODEL_YOLO_INIT = "best_kaggle.pt"                     # poids pré-entraînés YOLOv8s
CLASSIF_PATH    = "best_model_final.pth"
DATASET_DIR     = "yolo_dataset_preprocess"        # dossier dataset YOLO généré
YAML_PATH       = "nodule_dataset.yaml"
YOLO_RUN_NAME   = "nodule_finetuned"

# Modèle YOLO externe pré-entraîné sur nodules (LIDC, LUNA16, etc.)
# Mettre None pour désactiver le bootstrapping externe.
YOLO_EXTERNE_PATH = 'best_kaggle.pt'

# Seuil de confiance élevé pour le bootstrapping externe.
CONF_BOOTSTRAP = 0.50

# Seuils pipeline final
SEUIL_CONF_CLASS    = 0.38
SEUIL_CONF_LOCAL_LO = 0.01   # si classifieur dit POSITIF
SEUIL_RATTRAPAGE    = 0.40   # si classifieur dit NÉGATIF
SEUIL_AUGMENTATION  = 0.65   # conf YOLO pour augmenter le dataset (plus strict = moins de bruit)
SEUIL_DISTANCE_PX   = 50     # distance max (pixels) pour associer GT ↔ prédiction


# ─────────────────────────────────────────────
# 2. PREPROCESSING
# ─────────────────────────────────────────────
def thorax_mask(shape, margin=0.97, blur=51):
    h, w = shape
    y, x = np.ogrid[:h, :w]
    cy, cx = h / 2, w / 2
    ry, rx = (h / 2) * margin, (w / 2) * margin
    mask = ((y - cy) ** 2) / (ry ** 2) + ((x - cx) ** 2) / (rx ** 2)
    mask = (mask <= 1).astype(np.float32)
    mask = cv2.GaussianBlur(mask, (blur, blur), 0)
    return mask


def preprocess_image(img: np.ndarray, clip_percentiles=(1, 99)) -> np.ndarray:
    """Normalise, applique un masque thoracique, CLAHE et renvoie une image RGB uint8."""
    if len(img.shape) == 3:
        img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    img = img.astype(np.float32)
    p_low, p_high = np.percentile(img, clip_percentiles)
    img = np.clip(img, p_low, p_high)

    denom = p_high - p_low if (p_high - p_low) != 0 else 1e-6
    img = (img - p_low) / denom

    mask = thorax_mask(img.shape)
    background = np.percentile(img, 5)
    img = img * mask + background * (1 - mask)

    img = (img * 255).astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img = clahe.apply(img)
    img = cv2.GaussianBlur(img, (3, 3), sigmaX=0.5)
    return cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)


# ─────────────────────────────────────────────
# 3. SEPE PREPROCESSOR
# ─────────────────────────────────────────────
class SEPEPreprocessor:
    def __init__(self, sigma=1.0, alpha=1.5, sigma_e=0.1, lambda_reg=0.1):
        self.sigma = sigma
        self.alpha = alpha
        self.sigma_e = sigma_e
        self.lambda_reg = lambda_reg

    def __call__(self, image: np.ndarray) -> np.ndarray:
        I = image.astype(np.float32)
        G_sigma = gaussian_filter(I, sigma=self.sigma)
        grad_x = cv2.Sobel(I, cv2.CV_32F, 1, 0, ksize=3)
        grad_y = cv2.Sobel(I, cv2.CV_32F, 0, 1, ksize=3)
        grad_magnitude = np.sqrt(grad_x ** 2 + grad_y ** 2)
        W = np.exp(-(grad_magnitude ** 2) / (self.sigma_e ** 2))
        I_enh = I + self.alpha * W * (I - G_sigma)
        laplacian = cv2.Laplacian(I, cv2.CV_32F)
        I_SEPE = I_enh - self.lambda_reg * np.abs(laplacian)
        return np.clip(I_SEPE, I.min(), I.max())

    def batch_process(self, images: torch.Tensor) -> torch.Tensor:
        c = images.shape[1]
        kernel_size = int(6 * self.sigma) | 1
        x = torch.arange(kernel_size, dtype=torch.float32, device=images.device) - kernel_size // 2
        gauss = torch.exp(-x ** 2 / (2 * self.sigma ** 2))
        gauss = gauss / gauss.sum()
        kernel_2d = (gauss.unsqueeze(0) * gauss.unsqueeze(1)).expand(c, 1, -1, -1)
        padding = kernel_size // 2
        G_sigma = F.conv2d(images, kernel_2d, padding=padding, groups=c)

        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
                                dtype=torch.float32, device=images.device).view(1, 1, 3, 3).expand(c, 1, -1, -1)
        sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]],
                                dtype=torch.float32, device=images.device).view(1, 1, 3, 3).expand(c, 1, -1, -1)
        grad_x = F.conv2d(images, sobel_x, padding=1, groups=c)
        grad_y = F.conv2d(images, sobel_y, padding=1, groups=c)
        grad_mag = torch.sqrt(grad_x ** 2 + grad_y ** 2 + 1e-8)
        W = torch.exp(-(grad_mag ** 2) / (self.sigma_e ** 2))
        I_enh = images + self.alpha * W * (images - G_sigma)

        lap_k = torch.tensor([[0, 1, 0], [1, -4, 1], [0, 1, 0]],
                              dtype=torch.float32, device=images.device).view(1, 1, 3, 3).expand(c, 1, -1, -1)
        laplacian = F.conv2d(images, lap_k, padding=1, groups=c)
        return I_enh - self.lambda_reg * torch.abs(laplacian)


def _build_3ch_batch(images: torch.Tensor, sepe: SEPEPreprocessor) -> torch.Tensor:
    c1 = images[:, 0:1]
    c3 = images[:, 1:2]
    c2 = sepe.batch_process(c1)
    c2_max = c2.view(c2.size(0), -1).max(dim=1)[0].view(-1, 1, 1, 1)
    c2 = c2 / (c2_max + 1e-8)
    return torch.cat([c1, c2, c3], dim=1)


# ─────────────────────────────────────────────
# 4. ARCHITECTURE CLASSIFIEUR
# ─────────────────────────────────────────────
class MEAM(nn.Module):
    def __init__(self, channels: List[int], d_k: int = 64, d_v: int = 64):
        super().__init__()
        self.scales = len(channels)
        self.d_k = d_k
        self.q_proj = nn.ModuleList([nn.Conv2d(c, d_k, 1) for c in channels])
        self.k_proj = nn.ModuleList([nn.Conv2d(c, d_k, 1) for c in channels])
        self.v_proj = nn.ModuleList([nn.Conv2d(c, d_v, 1) for c in channels])
        self.output_proj = nn.Conv2d(d_v * self.scales, channels[-1], 1)
        self.norm = nn.BatchNorm2d(channels[-1])

    def forward(self, features: List[torch.Tensor]) -> torch.Tensor:
        target_size = features[-1].shape[2:]
        resized = [F.interpolate(f, size=target_size, mode='bilinear', align_corners=False)
                   if f.shape[2:] != target_size else f for f in features]
        Q = [p(f) for p, f in zip(self.q_proj, resized)]
        K = [p(f) for p, f in zip(self.k_proj, resized)]
        V = [p(f) for p, f in zip(self.v_proj, resized)]
        attended = []
        for i in range(self.scales):
            scores = [torch.bmm(Q[i].flatten(2).transpose(1, 2), K[j].flatten(2)) / math.sqrt(self.d_k)
                      for j in range(self.scales)]
            weights = [torch.softmax(s, dim=-1) for s in scores]
            agg = sum(torch.bmm(V[j].flatten(2), weights[j].transpose(1, 2)) for j in range(self.scales))
            attended.append(agg.view_as(V[i]))
        fused = torch.cat(attended, dim=1)
        return F.relu(self.norm(self.output_proj(fused)))


class EfficientBackbone(nn.Module):
    EXTRACT_LAYERS = [2, 3, 4, 6]
    _CHANNEL_MAP = {'s': [48, 64, 128, 256], 'm': [48, 80, 160, 304], 'l': [64, 96, 192, 384]}

    def __init__(self, variant='s', pretrained=True):
        super().__init__()
        self.extract_layers = self.EXTRACT_LAYERS
        self.channels = self._CHANNEL_MAP[variant]
        if variant == 's':
            from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights
            weights = EfficientNet_V2_S_Weights.IMAGENET1K_V1 if pretrained else None
            self.backbone = efficientnet_v2_s(weights=weights).features
        elif variant == 'm':
            from torchvision.models import efficientnet_v2_m, EfficientNet_V2_M_Weights
            weights = EfficientNet_V2_M_Weights.IMAGENET1K_V1 if pretrained else None
            self.backbone = efficientnet_v2_m(weights=weights).features
        else:
            from torchvision.models import efficientnet_v2_l, EfficientNet_V2_L_Weights
            weights = EfficientNet_V2_L_Weights.IMAGENET1K_V1 if pretrained else None
            self.backbone = efficientnet_v2_l(weights=weights).features

    def forward(self, x):
        features = []
        for i, block in enumerate(self.backbone):
            x = block(x)
            if i in self.extract_layers:
                features.append(x)
        return features, features[-1]


class NoduleClassifier(nn.Module):
    def __init__(self, backbone_variant='s'):
        super().__init__()
        self.backbone = EfficientBackbone(variant=backbone_variant)
        self.meam = MEAM(channels=self.backbone.channels)
        final_channels = self.backbone.channels[-1]
        self.classifier_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(final_channels, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        features_list, _ = self.backbone(x)
        fused = self.meam(features_list)
        return self.classifier_head(fused)


# ─────────────────────────────────────────────
# 5. WRAPPERS CLASSIFIEUR & LOCALISATEUR
# ─────────────────────────────────────────────
class Classifieur:
    def __init__(self, model_path=CLASSIF_PATH, device=device):
        self.device = device
        self.model = NoduleClassifier()
        state_dict = torch.load(model_path, map_location=device, weights_only=False)
        self.model.load_state_dict(state_dict)
        self.model.to(device).eval()

    def preprocess(self, img_np: np.ndarray) -> torch.Tensor:
        if len(img_np.shape) == 3 and img_np.shape[2] == 3:
            img_gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
        elif len(img_np.shape) == 3:
            img_gray = img_np[:, :, 0]
        else:
            img_gray = img_np

        if img_gray.dtype != np.uint8:
            mn, mx = img_gray.min(), img_gray.max()
            img_gray = ((img_gray - mn) / (mx - mn + 1e-8) * 255).astype(np.uint8)

        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        img_clahe = clahe.apply(img_gray)
        t_raw   = torch.from_numpy(img_gray).float() / 255.0
        t_clahe = torch.from_numpy(img_clahe).float() / 255.0
        t_input = torch.stack([t_raw, t_clahe], dim=0).unsqueeze(0).to(self.device)
        return _build_3ch_batch(t_input, SEPEPreprocessor())

    def pred(self, img_np: np.ndarray, threshold: float = SEUIL_CONF_CLASS) -> torch.Tensor:
        img_tensor = self.preprocess(img_np)
        with torch.no_grad():
            outputs = self.model(img_tensor)
            probs = F.softmax(outputs, dim=1)
            return (probs[:, 1] >= threshold).long()


class Localisation:
    def __init__(self, model_path: str, device=device):
        self.model = YOLO(model_path).to(device)

    def pred(self, img_np: np.ndarray, conf_threshold: float = 0.50) -> List[dict]:
        img_rgb = preprocess_image(img_np)
        # Letterbox explicite — cohérent avec le dataset d'entraînement
        img_lb, scale, pad_x, pad_y = _letterbox(img_rgb, YOLO_IMG_SIZE)
        h_orig, w_orig = img_rgb.shape[:2]

        # Bornes de la zone image réelle dans le canvas letterboxé (hors padding gris)
        x_min_v = pad_x
        y_min_v = pad_y
        x_max_v = pad_x + int(round(w_orig * scale))
        y_max_v = pad_y + int(round(h_orig * scale))

        # Image en niveaux de gris pour le filtre anatomique (colonne/os)
        img_gray_lb = cv2.cvtColor(img_lb, cv2.COLOR_RGB2GRAY).astype(np.float32)

        results = self.model(img_lb, conf=conf_threshold, verbose=False)
        nodules = []
        for box in results[0].boxes:
            conf = float(box.conf[0])
            if conf < conf_threshold:
                continue
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            cx = (x1 + x2) / 2.0
            cy = (y1 + y2) / 2.0

            # Rejette les détections dont le centre tombe dans la zone de padding
            if not (x_min_v <= cx <= x_max_v and y_min_v <= cy <= y_max_v):
                continue

            # ── Filtre anatomique : colonne vertébrale et os ─────────────────
            # La colonne est une bande verticale très lumineuse au centre.
            # Les os (côtes, clavicules) sont aussi très intenses.
            # On rejette les détections dont la région est trop lumineuse.
            rx1, ry1 = int(max(0, x1)), int(max(0, y1))
            rx2, ry2 = int(min(YOLO_IMG_SIZE, x2)), int(min(YOLO_IMG_SIZE, y2))
            if rx2 > rx1 and ry2 > ry1:
                patch = img_gray_lb[ry1:ry2, rx1:rx2]
                mean_intensity = float(patch.mean())
                # Seuil empirique : pixels > 200/255 = os/colonne (très blanc après CLAHE)
                if mean_intensity > 200:
                    continue

            # ── Reprojection vers l'espace image originale ───────────────────
            # Les coordonnées YOLO sont dans l'espace letterbox (0–640).
            # GT dans df_bbox est en pixels originaux → on reprojecte.
            cx_orig = (cx - pad_x) / scale
            cy_orig = (cy - pad_y) / scale

            nodules.append({
                'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2,
                'cx': cx_orig,   # espace image originale
                'cy': cy_orig,   # espace image originale
                'cx_lb': cx,     # espace letterbox (utile pour visualisation)
                'cy_lb': cy,
                'conf': conf
            })
        return nodules


# ─────────────────────────────────────────────
# 6. CHARGEMENT DES DONNÉES
# ─────────────────────────────────────────────
def load_data(csv_bbox=CSV_BBOX, csv_labels=CSV_LABELS):
    df_bbox = pd.read_csv(csv_bbox)

    labels = pd.read_csv(csv_labels)
    labels["label"] = (labels["label"] != "No Finding").astype(int)
    labels["path"] = np.where(
        labels["LIDC_ID"].isna(),
        "nih_filtered_images/",
        "lidc_png_16_bit/"
    ) + labels["file_name"]

    no_sane  = labels[labels["label"] == 1]
    annoted  = labels[labels["LIDC_ID"].notna()]
    return df_bbox, labels, no_sane, annoted


# ─────────────────────────────────────────────
# 7. DIAGNOSTIC DU DATASET
# ─────────────────────────────────────────────
def diagnose_dataset(df_bbox: pd.DataFrame, labels: pd.DataFrame,
                     out_dir: str,
                     n_samples: int = 8,
                     box_frac: float = 0.03):
    """
    Ouvre n_samples images déjà générées dans out_dir et dessine les
    boîtes GT en rouge. Sauvegarde les images annotées dans 'diagnostic/'.
    Lance ça AVANT d'entraîner pour vérifier que le dataset est sain.
    """
    diag_dir = "diagnostic"
    os.makedirs(diag_dir, exist_ok=True)

    img_dir_train = os.path.join(out_dir, "images", "train")
    if not os.path.exists(img_dir_train):
        print(f"  [DIAG] Dossier introuvable : {img_dir_train}")
        return

    sample_files = [f for f in os.listdir(img_dir_train) if f.endswith(".png")][:n_samples]
    if not sample_files:
        print("  [DIAG] Aucune image trouvée dans le dossier train.")
        return

    print(f"\n[DIAGNOSTIC] Vérification sur {len(sample_files)} images ─────────────")
    for fname in sample_files:
        img_path = os.path.join(img_dir_train, fname)
        lbl_path = os.path.join(out_dir, "labels", "train", fname.replace(".png", ".txt"))

        img = cv2.imread(img_path)
        if img is None:
            print(f"  [DIAG] Impossible de lire {img_path}")
            continue
        h, w = img.shape[:2]

        boxes_drawn = 0
        bw_px = bh_px = 0
        if os.path.exists(lbl_path):
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    _, cx_n, cy_n, bw_n, bh_n = map(float, parts)
                    cx_px = int(cx_n * w)
                    cy_px = int(cy_n * h)
                    bw_px = int(bw_n * w)
                    bh_px = int(bh_n * h)
                    x1 = max(0, cx_px - bw_px // 2)
                    y1 = max(0, cy_px - bh_px // 2)
                    x2 = min(w, cx_px + bw_px // 2)
                    y2 = min(h, cy_px + bh_px // 2)
                    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), 2)
                    cv2.circle(img, (cx_px, cy_px), 4, (0, 255, 0), -1)
                    boxes_drawn += 1
        else:
            print(f"  [DIAG] Pas de label pour {fname}")

        out_path = os.path.join(diag_dir, fname)
        cv2.imwrite(out_path, img)
        print(f"  {fname}  →  {boxes_drawn} boîte(s)  "
              f"(image {w}×{h}px, boîte GT ≈ {bw_px}×{bh_px}px)")

    print(f"\n  Annotations totales           : {len(df_bbox)}")
    print(f"  Images annotées uniques       : {df_bbox['file_name'].nunique()}")
    print(f"  Centre X — min={df_bbox['x'].min()}  max={df_bbox['x'].max()}  "
          f"moy={df_bbox['x'].mean():.0f}")
    print(f"  Centre Y — min={df_bbox['y'].min()}  max={df_bbox['y'].max()}  "
          f"moy={df_bbox['y'].mean():.0f}")
    print(f"\n  Images de diagnostic sauvegardées dans '{diag_dir}/'")
    print(f"  → Ouvre ces images pour vérifier que les croix rouges tombent bien sur les nodules.")
    print("[FIN DIAGNOSTIC] ────────────────────────────────────────────\n")


# ─────────────────────────────────────────────
# 7b. CALIBRATION DE LA TAILLE DE BOÎTE
# ─────────────────────────────────────────────
def estimate_box_frac(labels: pd.DataFrame, df_bbox: pd.DataFrame,
                      n_samples: int = 30) -> float:
    sample_fnames = df_bbox["file_name"].unique()[:n_samples]
    widths = []

    for fname in sample_fnames:
        row = labels[labels["file_name"] == fname]
        if row.empty:
            continue
        path_img = row.iloc[0]["path"]
        try:
            img_raw = Image.open(path_img)
            widths.append(img_raw.size[0])
        except Exception:
            continue

    if not widths:
        print("  [WARN] Impossible d'estimer la taille des images, box_frac=0.03 par défaut.")
        return 0.03

    median_w = np.median(widths)
    if median_w >= 1024:
        box_frac = 0.04
    elif median_w >= 512:
        box_frac = 0.03
    else:
        box_frac = 0.05

    print(f"  Largeur médiane des images : {median_w:.0f}px  →  box_frac={box_frac}")
    return box_frac


# ─────────────────────────────────────────────
# 8. GÉNÉRATION DU DATASET YOLO
# ─────────────────────────────────────────────
YOLO_IMG_SIZE = 640   # résolution d'entraînement YOLO
BOX_FRAC = 0.05


def _letterbox(img_rgb: np.ndarray, target: int) -> Tuple[np.ndarray, float, int, int]:
    """
    Redimensionne img_rgb dans un carré target×target avec letterboxing gris.
    Retourne (img_resized, scale, pad_x, pad_y).
    """
    h, w = img_rgb.shape[:2]
    scale = min(target / w, target / h)
    new_w = int(round(w * scale))
    new_h = int(round(h * scale))
    resized = cv2.resize(img_rgb, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    canvas = np.full((target, target, 3), 114, dtype=np.uint8)
    pad_x = (target - new_w) // 2
    pad_y = (target - new_h) // 2
    canvas[pad_y:pad_y + new_h, pad_x:pad_x + new_w] = resized
    return canvas, scale, pad_x, pad_y


def generate_yolo_dataset(df_bbox: pd.DataFrame, labels: pd.DataFrame,
                           out_dir: str = DATASET_DIR,
                           val_ratio: float = 0.2,
                           img_size: int = YOLO_IMG_SIZE,
                           box_frac: float = BOX_FRAC,
                           df_bbox_gt: pd.DataFrame = None):
    """
    df_bbox_gt : si fourni, le val set est tiré UNIQUEMENT de ces annotations
                 (annotations GT pures, sans pseudo-labels). Ça évite que la
                 validation soit polluée par du bruit semi-supervisé.
    """
    random.seed(SEED)
    np.random.seed(SEED)

    nodule_px = box_frac * img_size
    print(f"  img_size={img_size}px  |  box_frac={box_frac}  "
          f"→  boîte GT = {nodule_px:.1f}×{nodule_px:.1f}px dans l'image finale")

    for split in ("train", "val"):
        os.makedirs(os.path.join(out_dir, "images", split), exist_ok=True)
        os.makedirs(os.path.join(out_dir, "labels", split), exist_ok=True)

    # Val set : uniquement sur les annotations GT si df_bbox_gt fourni
    gt_fnames = df_bbox_gt["file_name"].unique().tolist() if df_bbox_gt is not None                 else df_bbox["file_name"].unique().tolist()
    random.shuffle(gt_fnames)
    n_val = max(1, int(len(gt_fnames) * val_ratio))
    val_set = set(gt_fnames[:n_val])

    file_names = df_bbox["file_name"].unique().tolist()
    random.shuffle(file_names)

    for fname in tqdm(file_names, desc="  Génération dataset", unit="img"):
        row_label = labels[labels["file_name"] == fname]
        if row_label.empty:
            continue
        path_img = row_label.iloc[0]["path"]
        split_name = "val" if fname in val_set else "train"

        try:
            img_np = np.array(Image.open(path_img))

            img_rgb = preprocess_image(img_np)
            img_lb, scale, pad_x, pad_y = _letterbox(img_rgb, img_size)

            img_out = os.path.join(out_dir, "images", split_name, fname)
            cv2.imwrite(img_out, cv2.cvtColor(img_lb, cv2.COLOR_RGB2BGR))

            rows_gt = df_bbox[df_bbox["file_name"] == fname]
            label_path = os.path.join(out_dir, "labels", split_name,
                                      fname.replace(".png", ".txt"))
            with open(label_path, "w") as lf:
                for _, r in rows_gt.iterrows():
                    cx_lb = r["x"] * scale + pad_x
                    cy_lb = r["y"] * scale + pad_y
                    cx_lb = max(0.0, min(img_size - 1, cx_lb))
                    cy_lb = max(0.0, min(img_size - 1, cy_lb))
                    cx_n  = cx_lb / img_size
                    cy_n  = cy_lb / img_size
                    lf.write(f"0 {cx_n:.6f} {cy_n:.6f} "
                             f"{box_frac:.6f} {box_frac:.6f}\n")

        except Exception as e:
            print(f"  [SKIP] {fname} : {e}")

    print(f"Dataset YOLO créé dans '{out_dir}'  "
          f"({len(file_names)-n_val} train / {n_val} val)")


def write_yaml(out_dir: str = DATASET_DIR, yaml_path: str = YAML_PATH):
    data = {
        "path":  os.path.abspath(out_dir),
        "train": "images/train",
        "val":   "images/val",
        "nc":    1,
        "names": ["Nodule"]
    }
    with open(yaml_path, "w") as f:
        yaml.dump(data, f, default_flow_style=False)
    print(f"YAML écrit : {yaml_path}")


# ─────────────────────────────────────────────
# 9. ENTRAÎNEMENT YOLO
# ─────────────────────────────────────────────
def train_yolo(yaml_path: str = YAML_PATH,
               weights: str = MODEL_YOLO_INIT,
               run_name: str = YOLO_RUN_NAME,
               epochs: int = 150,
               n_train_images: int = None,
               resume: bool = False) -> Tuple[str, float]:        # ← AJOUT
    """
    Entraîne YOLO et retourne (chemin_best.pt, mAP50_best).

    Si resume=True ET que runs/detect/{run_name}/weights/last.pt existe,
    reprend l'entraînement exactement là où il s'est arrêté
    (epoch, optimizer, scheduler, état early-stopping).
    Sinon démarre normalement depuis `weights`.
    """
    set_seed(SEED)

    if n_train_images is not None and n_train_images < 64:
        batch = 4
        print(f"  [INFO] Petit dataset ({n_train_images} imgs train) → batch=4")
    else:
        batch = 8

    # ── RESUME ──────────────────────────────────────────────────────────────
    last_pt = f"runs/detect/{run_name}/weights/last.pt"
    if resume and os.path.exists(last_pt):
        print(f"  [RESUME] Reprise depuis {last_pt}")
        model = YOLO(last_pt)
        model.train(resume=True)
    else:
        if resume:
            print(f"  [INFO] last.pt introuvable pour '{run_name}', démarrage normal.")
        model = YOLO(weights)
        model.train(
            data=yaml_path,
            epochs=epochs,
            imgsz=YOLO_IMG_SIZE,
            batch=batch,
            device=0 if torch.cuda.is_available() else "cpu",
            workers=0,
            name=run_name,
            seed=SEED,
            lr0=0.001,
            lrf=0.01,
            momentum=0.937,
            weight_decay=0.0005,
            warmup_epochs=5,
            warmup_momentum=0.8,
            patience=40,
            cache=True,
            hsv_h=0.0, hsv_s=0.0, hsv_v=0.3,
            degrees=10.0,    # rotation : nodules ne sont pas toujours droits
            translate=0.15,  # décalage spatial : casse le biais positionnel
            scale=0.3,       # zoom in/out
            shear=0.0, perspective=0.0,
            flipud=0.0,
            fliplr=0.5,      # symétrie G/D : double la diversité positionnelle
            mosaic=0.0,
            mixup=0.0, copy_paste=0.0,
        )
    # ────────────────────────────────────────────────────────────────────────

    best_pt = f"runs/detect/{run_name}/weights/best.pt"

    try:
        csv_path = f"runs/detect/{run_name}/results.csv"
        df_res = pd.read_csv(csv_path)
        df_res.columns = df_res.columns.str.strip()
        map50_col = [c for c in df_res.columns if "mAP50" in c and "95" not in c][0]
        best_map50 = float(df_res[map50_col].max())
    except Exception as e:
        print(f"  [WARN] Impossible de lire le mAP50 depuis le CSV : {e}")
        best_map50 = 0.0

    print(f"Meilleurs poids : {best_pt}  |  mAP50 max = {best_map50:.4f}")
    return best_pt, best_map50


# ─────────────────────────────────────────────
# 9b. AUGMENTATION SEMI-SUPERVISÉE
# ─────────────────────────────────────────────
MAP50_MIN_POUR_AUGMENTATION = 0.35  # relevé : évite les pseudo-labels trop bruités
DEDUP_RADIUS_PX = 30
MAX_BOXES_PAR_IMAGE = 2


def _dedup_boxes(annotations: List[dict], radius: int = DEDUP_RADIUS_PX) -> List[dict]:
    if not annotations:
        return []

    by_file: dict = {}
    for ann in annotations:
        by_file.setdefault(ann["file_name"], []).append(ann)

    kept = []
    for fname, dets in by_file.items():
        dets = sorted(dets, key=lambda d: d["conf"], reverse=True)
        suppressed = [False] * len(dets)
        for i, d in enumerate(dets):
            if suppressed[i]:
                continue
            kept.append(d)
            for j in range(i + 1, len(dets)):
                if suppressed[j]:
                    continue
                dist = math.dist((d["x"], d["y"]), (dets[j]["x"], dets[j]["y"]))
                if dist < radius:
                    suppressed[j] = True
    return kept


def augment_dataset(df_bbox: pd.DataFrame,
                    df_malades: pd.DataFrame,
                    yolo_model_path: str,
                    
                    model_map50: float,
                    conf_threshold: float = SEUIL_AUGMENTATION) -> pd.DataFrame:
    if model_map50 < MAP50_MIN_POUR_AUGMENTATION:
        print(f"  [SKIP augmentation] mAP50={model_map50:.4f} < "
              f"{MAP50_MIN_POUR_AUGMENTATION} → modèle trop faible, "
              f"on ne pollue pas le dataset.")
        return df_bbox

    max_boxes = 1 if model_map50 < 0.55 else MAX_BOXES_PAR_IMAGE  # plus conservateur

    df_bbox_copy = df_bbox.copy()
    if "path" not in df_bbox_copy.columns:
        df_bbox_copy["path"] = "lidc_png_16_bit/" + df_bbox_copy["file_name"]

    df_malades_sans_bbox = df_malades[
        ~df_malades["file_name"].isin(df_bbox["file_name"].unique())
    ]
    print(f"  Images malades sans annotation : {len(df_malades_sans_bbox)}")

    model_yolo = YOLO(yolo_model_path)
    nouvelles_annotations = []

    for _, row in tqdm(df_malades_sans_bbox.iterrows(), total=len(df_malades_sans_bbox), desc="  Augmentation", unit="img"):
        path_img = row["path"]
        try:
            img_np  = np.array(Image.open(path_img))
            img_rgb = preprocess_image(img_np)
            img_lb, scale, pad_x, pad_y = _letterbox(img_rgb, YOLO_IMG_SIZE)
            results  = model_yolo(img_lb, augment=True, verbose=False)

            dets_img = []
            for box in results[0].boxes:
                conf   = float(box.conf[0])
                cls_id = int(box.cls[0])
                if conf >= conf_threshold and cls_id == 0:
                    x1, y1, x2, y2 = box.xyxy[0].tolist()
                    cx_orig = ((x1 + x2) / 2 - pad_x) / scale
                    cy_orig = ((y1 + y2) / 2 - pad_y) / scale
                    dets_img.append({
                        "file_name": row["file_name"],
                        "x":   round(cx_orig),
                        "y":   round(cy_orig),
                        "path": path_img,
                        "conf": conf,
                    })

            dets_img.sort(key=lambda d: d["conf"], reverse=True)
            nouvelles_annotations.extend(dets_img[:max_boxes])

        except Exception:
            continue

    if not nouvelles_annotations:
        print("  Aucune nouvelle annotation générée.")
        return df_bbox_copy

    nouvelles_annotations = _dedup_boxes(nouvelles_annotations)

    df_new = pd.DataFrame(nouvelles_annotations)
    n_avant = len(df_new)

    df_enrichi = pd.concat(
        [df_bbox_copy, df_new[["file_name", "path", "x", "y"]]],
        ignore_index=True
    )
    print(f"  {n_avant} nouvelles boîtes (après dédup) "
          f"→  total {len(df_enrichi)} annotations.")
    return df_enrichi


# ─────────────────────────────────────────────
# 10. BOUCLE D'ENTRAÎNEMENT ITÉRATIVE
# ─────────────────────────────────────────────
def iterative_training(df_bbox_init: pd.DataFrame,
                       labels: pd.DataFrame,
                       no_sane: pd.DataFrame,

                       n_iterations: int = 3,
                       epochs_per_iter: int = 150,
                       resume_iter: int = None):          # ← AJOUT
    """
    Boucle :
      1. Génère dataset YOLO
      2. Diagnostic visuel à l'itération 1
      3. Entraîne YOLO
      4. Augmente le dataset si mAP50 suffisant
      5. Recommence jusqu'à convergence (ou n_iterations)

    Paramètre resume_iter
    ---------------------
    None  → démarre normalement depuis l'itération 1
    N     → reprend à l'itération N :
              • charge localization_labels_iter{N-1}.csv si N > 1
              • utilise best.pt de l'itération N-1 comme point de départ
              • si last.pt de l'itération N existe → reprend depuis ce checkpoint
              • si le dossier dataset existe déjà → régénération sautée
    """
    # ── Point de départ selon resume_iter ───────────────────────────────────
    if resume_iter is None:
        resume_iter = 1

    if resume_iter > 1:
        csv_prev = f"localization_labels_iter{resume_iter - 1}.csv"
        if os.path.exists(csv_prev):
            df_bbox = pd.read_csv(csv_prev)
            print(f"  [RESUME] Dataset chargé depuis {csv_prev} ({len(df_bbox)} annotations)")
        else:
            print(f"  [RESUME] {csv_prev} introuvable → dataset initial utilisé.")
            df_bbox = df_bbox_init.copy()

        prev_best = f"runs/detect/{YOLO_RUN_NAME}_iter{resume_iter - 1}/weights/best.pt"
        best_model_path = prev_best if os.path.exists(prev_best) else MODEL_YOLO_INIT
        print(f"  [RESUME] Poids de départ : {best_model_path}")
    else:
        df_bbox = df_bbox_init.copy()
        best_model_path = MODEL_YOLO_INIT
    # ────────────────────────────────────────────────────────────────────────

    prev_map50 = 0.0

    for iteration in range(resume_iter, n_iterations + 1):
        print(f"\n{'='*55}")
        print(f"  ITÉRATION {iteration}/{n_iterations}")
        print(f"{'='*55}")

        run_name  = f"{YOLO_RUN_NAME}_iter{iteration}"
        out_dir   = f"{DATASET_DIR}_iter{iteration}"
        is_resume = (iteration == resume_iter and resume_iter > 1)

        # Génération dataset (sautée si reprise et dossier déjà présent)
        if is_resume and os.path.exists(os.path.join(out_dir, "images", "train")):
            print(f"  [RESUME] Dataset existant dans '{out_dir}', génération sautée.")
            write_yaml(out_dir=out_dir, yaml_path=YAML_PATH)
        else:
            # df_bbox_init contient uniquement les GT originaux → val set toujours propre
            generate_yolo_dataset(df_bbox, labels,
                                  out_dir=out_dir, df_bbox_gt=df_bbox_init)
            write_yaml(out_dir=out_dir, yaml_path=YAML_PATH)

        # Diagnostic uniquement à l'itération 1 (pas en mode reprise)
        if iteration == 1 and not is_resume:
            diagnose_dataset(df_bbox, labels, out_dir=out_dir)

        n_train = len([f for f in os.listdir(os.path.join(out_dir, "images", "train"))
                       if f.endswith(".png")])
        n_val   = len([f for f in os.listdir(os.path.join(out_dir, "images", "val"))
                       if f.endswith(".png")])
        print(f"  Images train : {n_train}  |  Images val : {n_val}")

        # Entraînement — resume=True si on reprend l'itération interrompue
        best_model_path, current_map50 = train_yolo(
            yaml_path=YAML_PATH,
            weights=best_model_path,
            run_name=run_name,
            epochs=epochs_per_iter,
            n_train_images=n_train,
            resume=is_resume,                                      # ← AJOUT
        )

        delta = current_map50 - prev_map50
        print(f"  ΔmAP50 = {delta:+.4f}  (prev={prev_map50:.4f} → now={current_map50:.4f})")
        if iteration > 1 and delta < 0.01:
            print("  → Convergence atteinte, arrêt anticipé de la boucle.")
            break
        prev_map50 = current_map50

        df_bbox = augment_dataset(
            df_bbox, no_sane, best_model_path,
            model_map50=current_map50,
        )
        df_bbox.to_csv(f"localization_labels_iter{iteration}.csv", index=False)

    print("\nBoucle itérative terminée.")
    print(f"Meilleur modèle final : {best_model_path}")
    return best_model_path, df_bbox


# ─────────────────────────────────────────────
# 11. RECHERCHE DES MEILLEURS SEUILS YOLO
# ─────────────────────────────────────────────
def _load_image_from_disk(path_img: str) -> np.ndarray:
    """Charge une image depuis le disque (path relatif au répertoire courant)."""
    img = Image.open(path_img)
    return np.array(img)


def find_best_thresholds(df_bbox: pd.DataFrame,
                         df_labels: pd.DataFrame,
                         yolo_model_path: str,
                         classifier: Classifieur,
                         conf_values: List[float] = None,
                         rattrapage_values: List[float] = None) -> dict:
    if conf_values is None:
        conf_values = [0.01, 0.05, 0.10, 0.20, 0.30, 0.40, 0.50]
    if rattrapage_values is None:
        rattrapage_values = [0.10, 0.15, 0.20, 0.25, 0.30]

    localisateur = Localisation(model_path=yolo_model_path, device=device)

    # Pré-charge toutes les images en mémoire (évite de rouvrir les zips en boucle)
    print("  Chargement des images pour le grid-search...")
    images_cache: dict = {}
    for nom_fichier in tqdm(df_bbox["file_name"].unique(), desc="  Chargement cache", unit="img"):
        row = df_labels[df_labels["file_name"] == nom_fichier]
        if row.empty:
            continue
        path_img = row.iloc[0]["path"]
        try:
            images_cache[nom_fichier] = (path_img,
                                         _load_image_from_disk(path_img))
        except Exception:
            continue
    print(f"  {len(images_cache)} images chargées.")

    best_f1    = -1
    best_cfg   = {}
    results_grid = []

    grid = [(lo, rt) for lo in conf_values for rt in rattrapage_values]
    for conf_lo, conf_rt in tqdm(grid, desc="  Grid-search seuils", unit="cfg"):
        tp = fp = fn = 0

        for nom_fichier, (path_img, img_np) in images_cache.items():
            lignes_gt   = df_bbox[df_bbox["file_name"] == nom_fichier]
            boites_vraies = [{"x": r["x"], "y": r["y"]} for _, r in lignes_gt.iterrows()]

            try:
                pred_class = classifier.pred(img_np, threshold=SEUIL_CONF_CLASS).item()

                if pred_class == 1:
                    nodules_trouves = localisateur.pred(img_np, conf_threshold=conf_lo)
                else:
                    nodules_rattrapage = localisateur.pred(img_np, conf_threshold=conf_rt)
                    nodules_trouves    = nodules_rattrapage if nodules_rattrapage else []

                vraies_restantes = list(range(len(boites_vraies)))
                for pred in nodules_trouves:
                    best_dist, best_idx = float("inf"), -1
                    for j in vraies_restantes:
                        d = math.dist((pred["cx"], pred["cy"]),
                                      (boites_vraies[j]["x"], boites_vraies[j]["y"]))
                        if d < best_dist:
                            best_dist, best_idx = d, j
                    if best_dist < SEUIL_DISTANCE_PX and best_idx != -1:
                        tp += 1
                        vraies_restantes.remove(best_idx)
                    else:
                        fp += 1
                fn += len(vraies_restantes)

            except Exception:
                continue

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

        results_grid.append({
            "conf_local": conf_lo, "conf_rattrapage": conf_rt,
            "tp": tp, "fp": fp, "fn": fn,
            "precision": round(precision, 4),
            "recall":    round(recall, 4),
            "f1":        round(f1, 4)
        })

        if f1 > best_f1:
            best_f1  = f1
            best_cfg = {"conf_local": conf_lo, "conf_rattrapage": conf_rt,
                        "precision": precision, "recall": recall, "f1": f1}

    df_grid = pd.DataFrame(results_grid).sort_values("f1", ascending=False)
    df_grid.to_csv("threshold_grid_search.csv", index=False)
    print("\n--- MEILLEURE CONFIGURATION ---")
    for k, v in best_cfg.items():
        print(f"  {k:20s}: {v}")
    return best_cfg, df_grid


# ─────────────────────────────────────────────
# 12. ÉVALUATION FINALE DU PIPELINE
# ─────────────────────────────────────────────
def evaluate_distance(df_bbox: pd.DataFrame,
                      df_labels: pd.DataFrame,
                      classifier: Classifieur,
                      localisateur: Localisation,
                      seuil_conf_class: float = SEUIL_CONF_CLASS,
                      seuil_conf_local: float = SEUIL_CONF_LOCAL_LO,
                      seuil_rattrapage: float = SEUIL_RATTRAPAGE):

    images_evaluees = images_detectees = 0
    nodules_vrais_total = nodules_predits_total = 0
    nodules_en_trop = nodules_en_moins = 0
    distances_centres = []

    for nom_fichier in tqdm(df_bbox["file_name"].unique(), desc="  Évaluation", unit="img"):
        lignes_labels = df_labels[df_labels["file_name"] == nom_fichier]
        if lignes_labels.empty:
            continue
        path_img = lignes_labels.iloc[0]["path"]

        lignes_gt     = df_bbox[df_bbox["file_name"] == nom_fichier]
        boites_vraies = [{"x": r["x"], "y": r["y"]} for _, r in lignes_gt.iterrows()]

        try:
            img_np = _load_image_from_disk(path_img)
            prediction = classifier.pred(img_np, threshold=seuil_conf_class)
            nodules_trouves = []
            images_evaluees += 1
            nodules_vrais_total += len(boites_vraies)

            if prediction.item() == 1:
                images_detectees += 1
                nodules_trouves = localisateur.pred(img_np, conf_threshold=seuil_conf_local)
            else:
                nodules_rattrapage = localisateur.pred(img_np, conf_threshold=seuil_rattrapage)
                if nodules_rattrapage:
                    images_detectees += 1
                    nodules_trouves = nodules_rattrapage

            nodules_predits_total += len(nodules_trouves)
            vraies_non_assignees  = list(range(len(boites_vraies)))
            predites_non_assignees = list(range(len(nodules_trouves)))

            for i, pred in enumerate(nodules_trouves):
                best_dist, best_idx = float("inf"), -1
                for j in vraies_non_assignees:
                    d = math.dist((pred["cx"], pred["cy"]),
                                  (boites_vraies[j]["x"], boites_vraies[j]["y"]))
                    if d < best_dist:
                        best_dist, best_idx = d, j
                if best_dist < SEUIL_DISTANCE_PX and best_idx != -1:
                    distances_centres.append(best_dist)
                    vraies_non_assignees.remove(best_idx)
                    predites_non_assignees.remove(i)

            nodules_en_moins += len(vraies_non_assignees)
            nodules_en_trop  += len(predites_non_assignees)

        except Exception as e:
            print(f"  Erreur sur {path_img} : {e}")
            continue

    dist_moy = np.mean(distances_centres) if distances_centres else 0
    sensibilite = images_detectees / images_evaluees if images_evaluees > 0 else 0
    tp = len(distances_centres)
    fp = nodules_en_trop
    fn = nodules_en_moins
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    print("\n--- RÉSULTATS PIPELINE SUR IMAGES ANNOTÉES ---")
    print(f"  Images évaluées                    : {images_evaluees}")
    print(f"  Images franchissant le classifieur  : {images_detectees}  (Sensibilité: {sensibilite:.2%})")
    print("\n--- RÉSULTATS LOCALISATION ---")
    print(f"  GT nodules                          : {nodules_vrais_total}")
    print(f"  Nodules prédits                     : {nodules_predits_total}")
    print(f"  TP (correctement localisés)         : {tp}")
    print(f"  FP (en trop)                        : {fp}")
    print(f"  FN (manqués)                        : {fn}")
    print(f"  Précision                           : {precision:.4f}")
    print(f"  Rappel                              : {recall:.4f}")
    print(f"  F1                                  : {f1:.4f}")
    print(f"  Distance moyenne centre-à-centre    : {dist_moy:.2f} px")

    return {
        "images_evaluees": images_evaluees, "sensibilite_image": sensibilite,
        "tp": tp, "fp": fp, "fn": fn,
        "precision": precision, "recall": recall, "f1": f1,
        "distance_moyenne": dist_moy
    }


# ─────────────────────────────────────────────
# 13. BOOTSTRAPPING PAR MODÈLE EXTERNE
# ─────────────────────────────────────────────
def bootstrap_with_external_model(
        df_bbox: pd.DataFrame,
        df_malades: pd.DataFrame,

        external_model_path: str,
        conf_threshold: float = CONF_BOOTSTRAP,
        max_boxes_par_image: int = 3) -> pd.DataFrame:
    if external_model_path is None or not os.path.exists(external_model_path):
        print("[BOOTSTRAP] Aucun modèle externe trouvé, étape ignorée.")
        return df_bbox

    print(f"\n[BOOTSTRAP] Modèle externe : {external_model_path}")
    print(f"  Seuil de confiance : {conf_threshold}  |  Max boîtes/image : {max_boxes_par_image}")

    df_sans_bbox = df_malades[
        ~df_malades["file_name"].isin(df_bbox["file_name"].unique())
    ]
    print(f"  Images cibles (malades sans GT) : {len(df_sans_bbox)}")

    model_ext = YOLO(external_model_path)
    nouvelles = []

    for _, row in tqdm(df_sans_bbox.iterrows(), total=len(df_sans_bbox), desc="  Bootstrap", unit="img"):
        path_img = row["path"]
        try:
            img_np = np.array(Image.open(path_img))

            if img_np.dtype != np.uint8:
                mn, mx = img_np.min(), img_np.max()
                img_raw = ((img_np - mn) / (mx - mn + 1e-8) * 255).astype(np.uint8)
            else:
                img_raw = img_np
            if img_raw.ndim == 2:
                img_raw = cv2.cvtColor(img_raw, cv2.COLOR_GRAY2RGB)
            elif img_raw.shape[2] != 3:
                img_raw = cv2.cvtColor(img_raw, cv2.COLOR_GRAY2RGB)
            img_lb, scale, pad_x, pad_y = _letterbox(img_raw, YOLO_IMG_SIZE)

            results = model_ext(img_lb, conf=conf_threshold,
                                augment=True, verbose=False)

            dets = []
            for box in results[0].boxes:
                conf   = float(box.conf[0])
                cls_id = int(box.cls[0])
                if conf >= conf_threshold and cls_id == 0:
                    x1, y1, x2, y2 = box.xyxy[0].tolist()
                    cx_orig = ((x1 + x2) / 2 - pad_x) / scale
                    cy_orig = ((y1 + y2) / 2 - pad_y) / scale
                    dets.append({
                        "file_name": row["file_name"],
                        "x":   round(cx_orig),
                        "y":   round(cy_orig),
                        "path": path_img,
                        "conf": conf,
                    })

            dets.sort(key=lambda d: d["conf"], reverse=True)
            nouvelles.extend(dets[:max_boxes_par_image])

        except Exception:
            continue

    if not nouvelles:
        print("  [BOOTSTRAP] Aucune détection au seuil demandé.")
        return df_bbox

    nouvelles = _dedup_boxes(nouvelles, radius=DEDUP_RADIUS_PX)
    df_new    = pd.DataFrame(nouvelles)

    df_bbox_copy = df_bbox.copy()
    if "path" not in df_bbox_copy.columns:
        df_bbox_copy["path"] = "lidc_png_16_bit/" + df_bbox_copy["file_name"]

    df_enrichi = pd.concat(
        [df_bbox_copy, df_new[["file_name", "path", "x", "y"]]],
        ignore_index=True
    )
    print(f"  [BOOTSTRAP] {len(df_new)} pseudo-labels ajoutés "
          f"→ total {len(df_enrichi)} annotations.")
    df_enrichi.to_csv("localization_labels_bootstrap.csv", index=False)
    return df_enrichi



# ─────────────────────────────────────────────
# 15. VISUALISATION DES PRÉDICTIONS
# ─────────────────────────────────────────────
def visualize_predictions(
        df_bbox: pd.DataFrame,
        df_labels: pd.DataFrame,
        classifier: "Classifieur",
        n_malades:  int   = 500,
        n_saines:   int   = 500,
        n_display:  int   = 16,
        seuil_conf_class: float = SEUIL_CONF_CLASS,
        save_path: str = "visualisation_predictions.png",
) -> dict:
    """
    Evalue le CLASSIFIEUR sur n_malades images (label=1, tirées de df_labels)
    + n_saines images (label=0), indépendamment de df_bbox.

    Métriques : TP/FP/FN/TN au niveau image (classification binaire).
      TP = malade  prédit positif
      FP = saine   prédite positive
      FN = malade  prédit négatif
      TN = saine   prédite négative

    Affiche une grille de n_display images malades avec :
      + croix rouge  = position GT du nodule (si annotée dans df_bbox)
      Titre : Clf OK/NOK  |  résultat correct ou non
    """
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches

    random.seed(SEED)

    # 1. Echantillonnage depuis df_labels (pas seulement df_bbox)
    df_malades = df_labels[df_labels["label"] == 1]
    fnames_malades = df_malades["file_name"].tolist()
    random.shuffle(fnames_malades)
    fnames_malades = fnames_malades[:n_malades]

    df_saines = df_labels[df_labels["label"] == 0]
    fnames_saines = df_saines["file_name"].tolist()
    random.shuffle(fnames_saines)
    fnames_saines = fnames_saines[:n_saines]

    print(f"\n[VISU] Classification sur {len(fnames_malades)} malades + {len(fnames_saines)} saines")

    # 2. Inference classifieur uniquement
    tp = fp = fn = tn = 0
    display_data = []

    all_fnames = [(f, 1) for f in fnames_malades] + [(f, 0) for f in fnames_saines]

    for fname, gt_label in tqdm(all_fnames, desc="  Classification", unit="img"):
        row_lbl = df_labels[df_labels["file_name"] == fname]
        if row_lbl.empty:
            continue
        path_img = row_lbl.iloc[0]["path"]

        try:
            img_np = _load_image_from_disk(path_img)
        except Exception:
            continue

        pred_class = classifier.pred(img_np, threshold=seuil_conf_class).item()

        # Matrice de confusion image-level
        if gt_label == 1 and pred_class == 1:
            tp += 1
        elif gt_label == 0 and pred_class == 1:
            fp += 1
        elif gt_label == 1 and pred_class == 0:
            fn += 1
        else:
            tn += 1

        # GT nodules si disponibles (juste pour l'affichage)
        gt_pts = []
        rows_gt = df_bbox[df_bbox["file_name"] == fname]
        if not rows_gt.empty:
            gt_pts = [{"x": int(r["x"]), "y": int(r["y"])} for _, r in rows_gt.iterrows()]

        correct = (pred_class == gt_label)
        if gt_label == 1 and len(display_data) < n_display:
            display_data.append({
                "fname":      fname,
                "img_np":     img_np,
                "gt_pts":     gt_pts,
                "pred_class": pred_class,
                "correct":    correct,
            })

    # 3. Métriques classification
    total     = tp + fp + fn + tn
    accuracy  = (tp + tn) / total if total > 0 else 0
    precision = tp / (tp + fp)    if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn)    if (tp + fn) > 0 else 0
    f1        = 2*precision*recall / (precision+recall) if (precision+recall) > 0 else 0
    specificity = tn / (tn + fp)  if (tn + fp) > 0 else 0

    print(f"\n  Matrice de confusion (image-level) :")
    print(f"    TP={tp}  FP={fp}")
    print(f"    FN={fn}  TN={tn}")
    print(f"  Accuracy    : {accuracy:.3f}")
    print(f"  Precision   : {precision:.3f}")
    print(f"  Rappel      : {recall:.3f}  (sensibilite)")
    print(f"  Specificite : {specificity:.3f}")
    print(f"  F1          : {f1:.3f}")

    # 4. Grille matplotlib (images malades uniquement)
    n_cols = 4
    n_rows = math.ceil(len(display_data) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(n_cols * 4, n_rows * 4),
                             facecolor="#111111")
    axes = np.array(axes).flatten()

    for ax, data in zip(axes, display_data):
        img_rgb = preprocess_image(data["img_np"])
        img_lb, scale, pad_x, pad_y = _letterbox(img_rgb, YOLO_IMG_SIZE)
        ax.imshow(img_lb)
        ax.axis("off")

        # Croix rouge = GT nodule si annoté dans df_bbox
        for gt in data["gt_pts"]:
            cx_lb = gt["x"] * scale + pad_x
            cy_lb = gt["y"] * scale + pad_y
            s = 20
            ax.plot([cx_lb - s, cx_lb + s], [cy_lb,     cy_lb    ], color="#ff3333", lw=2.5)
            ax.plot([cx_lb,     cx_lb    ], [cy_lb - s, cy_lb + s], color="#ff3333", lw=2.5)

        # Titre : résultat classifieur
        clf_sym   = "POSITIF" if data["pred_class"] == 1 else "NEGATIF"
        title_col = "#aaffaa" if data["correct"] else "#ff6666"
        status    = "OK" if data["correct"] else "ERREUR"
        has_gt    = "annotee" if data["gt_pts"] else "non annotee"
        ax.set_title(
            data["fname"][:22] + "\n" + "Clf:{} [{}] {}".format(clf_sym, status, has_gt),
            fontsize=7, color=title_col, pad=3
        )

        # Bordure colorée selon correction
        for spine in ax.spines.values():
            spine.set_edgecolor("#aaffaa" if data["correct"] else "#ff6666")
            spine.set_linewidth(2)
            spine.set_visible(True)

    for ax in axes[len(display_data):]:
        ax.set_facecolor("#111111")
        ax.axis("off")

    legend_elems = [
        mpatches.Patch(color="#ff3333", label="GT nodule (croix)"),
        mpatches.Patch(color="#aaffaa", label="Classification correcte"),
        mpatches.Patch(color="#ff6666", label="Classification incorrecte"),
    ]
    fig.legend(handles=legend_elems, loc="lower center", ncol=3,
               fontsize=9, framealpha=0.85, bbox_to_anchor=(0.5, 0.0))

    suptitle = (
        "Classification  --  {} malades / {} saines  (seuil={:.2f})\n"
        "Acc={:.2f}  P={:.2f}  R={:.2f}  F1={:.2f}  Spec={:.2f}"
    ).format(len(fnames_malades), len(fnames_saines), seuil_conf_class,
             accuracy, precision, recall, f1, specificity)
    fig.suptitle(suptitle, fontsize=11, color="white", y=1.01)

    plt.tight_layout(rect=[0, 0.04, 1, 1])
    plt.savefig(save_path, dpi=130, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close()
    print(f"  Grille sauvegardee -> {save_path}")

    return {
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
        "accuracy": accuracy, "precision": precision,
        "recall": recall, "specificity": specificity, "f1": f1,
    }



# ─────────────────────────────────────────────
# 16. SPLIT TRAIN / VAL / TEST PROPRE
# ─────────────────────────────────────────────
def make_test_split(df_bbox: pd.DataFrame,
                    test_ratio: float = 0.15,
                    save_path: str = "test_split_fnames.txt") -> tuple:
    """
    Isole un test set AVANT tout entraînement.
    À appeler UNE SEULE FOIS et sauvegarder — ne jamais toucher ensuite.

    Retourne df_bbox_train (à utiliser pour iterative_training)
    et sauvegarde les noms de fichiers test dans save_path.

    Usage :
        df_bbox_train = make_test_split(df_bbox, test_ratio=0.15)
        # puis passer df_bbox_train à iterative_training
        # et évaluer sur df_bbox_test à la fin
    """
    random.seed(SEED)
    fnames = df_bbox["file_name"].unique().tolist()
    random.shuffle(fnames)

    n_test = max(1, int(len(fnames) * test_ratio))
    test_fnames  = set(fnames[:n_test])
    train_fnames = set(fnames[n_test:])

    df_bbox_test  = df_bbox[df_bbox["file_name"].isin(test_fnames)].copy()
    df_bbox_train = df_bbox[df_bbox["file_name"].isin(train_fnames)].copy()

    # Sauvegarde des noms test pour pouvoir les recharger
    with open(save_path, "w") as f:
        for fname in sorted(test_fnames):
            f.write(fname + "\n")

    print(f"Test split créé :")
    print(f"  Train : {len(df_bbox_train)} annotations  ({len(train_fnames)} images)")
    print(f"  Test  : {len(df_bbox_test)}  annotations  ({len(test_fnames)} images)")
    print(f"  Noms test sauvegardés → {save_path}")
    print(f"  !! Ne jamais utiliser ces images pendant l'entraînement !!")
    return df_bbox_train, df_bbox_test


def load_test_split(df_bbox: pd.DataFrame,
                    save_path: str = "test_split_fnames.txt"):
    """Recharge le test set depuis le fichier sauvegardé."""
    with open(save_path) as f:
        test_fnames = set(l.strip() for l in f if l.strip())
    df_bbox_test  = df_bbox[df_bbox["file_name"].isin(test_fnames)].copy()
    df_bbox_train = df_bbox[~df_bbox["file_name"].isin(test_fnames)].copy()
    print(f"Test split chargé : {len(df_bbox_test)} annotations test  |  {len(df_bbox_train)} train")
    return df_bbox_train, df_bbox_test

# ─────────────────────────────────────────────
# 14. MAIN
# ─────────────────────────────────────────────
if __name__ == "__main__":
    set_seed(SEED)

    df_bbox, labels, no_sane, annoted = load_data()
    print(f"Annotations GT initiales       : {len(df_bbox)}")
    print(f"Images positives (no_sane)     : {len(no_sane)}")
    print(f"Images annotées LIDC           : {len(annoted)}")
    final_model_path = None

    # ── Test split (à faire UNE FOIS avant tout entraînement) ───────────────
    # Première exécution : créer le split et sauvegarder
    df_bbox_train, df_bbox_test = make_test_split(df_bbox, test_ratio=0.15)
    # Exécutions suivantes : recharger le même split
    #   df_bbox_train, df_bbox_test = load_test_split(df_bbox)
    # Puis passer df_bbox_train à iterative_training et bootstrap,
    # et évaluer uniquement sur df_bbox_test à la fin.
    # ────────────────────────────────────────────────────────────────────────

   
    # ── Étape 0 : Bootstrapping par modèle externe (optionnel) ──────────────
    """df_bbox = bootstrap_with_external_model(
        df_bbox             = df_bbox,
        df_malades          = no_sane,
        external_model_path = YOLO_EXTERNE_PATH,
        conf_threshold      = CONF_BOOTSTRAP,
    )"""

    # ── Étape 1 : Boucle d'entraînement itérative ───────────────────────────
    # ┌─────────────────────────────────────────────────────────────────────┐
    # │  RESUME_ITERATION :                                                 │
    # │    None  →  démarre depuis le début                                 │
    # │    N     →  charge iter{N-1}.csv + reprend depuis last.pt iter N    │
    # └─────────────────────────────────────────────────────────────────────┘
    RESUME_ITERATION = None   # ← CHANGE ICI SI TU REPRENDS UN RUN

    final_model_path, df_bbox_final = iterative_training(
        df_bbox_init    = df_bbox_train,
        labels          = labels,
        no_sane         = no_sane,
        n_iterations    = 5,
        epochs_per_iter = 100,
        resume_iter     = RESUME_ITERATION,
    )
    df_bbox_final.to_csv("localization_labels_enrichi.csv", index=False)

    # ── Étape 2 : Recherche des meilleurs seuils ─────────────────────────────
    classifier = Classifieur(model_path=CLASSIF_PATH, device=device)
    """best_cfg, df_grid = find_best_thresholds(
        df_bbox         = df_bbox_final,
        df_labels       = labels,
        yolo_model_path = final_model_path,
        classifier      = classifier,
    )"""
    

    if not final_model_path:
        df_bbox_final = pd.read_csv("localization_labels_enrichi.csv")
        final_model_path = "runs/detect/nodule_finetuned_iter1/weights/best.pt"

    classifier   = Classifieur(model_path=CLASSIF_PATH, device=device)
    localisateur = Localisation(model_path=final_model_path, device=device)

    # ── Étape 3 : Évaluation finale ──────────────────────────────────────────
    metrics = evaluate_distance(
        df_bbox          = df_bbox_test,
        df_labels        = labels,
        classifier       = classifier,
        localisateur     = localisateur,
        seuil_conf_class = SEUIL_CONF_CLASS,
        seuil_conf_local = 0.1,
        seuil_rattrapage = 0.4,
    )

    # ── Étape 4 : Visualisation ──────────────────────────────────────────────
    visualize_predictions(
        df_bbox          = df_bbox_test,
        df_labels        = labels,
        classifier       = classifier,
        n_malades        = 500,
        n_saines         = 500,
        n_display        = 16,
        seuil_conf_class = SEUIL_CONF_CLASS,
        save_path        = "visualisation_predictions.png",
    )

    print("\nPipeline terminé.")

Device : cuda
Annotations GT initiales       : 179
Images positives (no_sane)     : 2818
Images annotées LIDC           : 113
Test split créé :
  Train : 153 annotations  (97 images)
  Test  : 26  annotations  (16 images)
  Noms test sauvegardés → test_split_fnames.txt
  !! Ne jamais utiliser ces images pendant l'entraînement !!

  ITÉRATION 1/5
  img_size=640px  |  box_frac=0.05  →  boîte GT = 32.0×32.0px dans l'image finale


  Génération dataset: 100%|████████████████████████████████████████████████████████████| 97/97 [00:11<00:00,  8.17img/s]


Dataset YOLO créé dans 'yolo_dataset_preprocess_iter1'  (78 train / 19 val)
YAML écrit : nodule_dataset.yaml

[DIAGNOSTIC] Vérification sur 8 images ─────────────
  0184.png  →  1 boîte(s)  (image 640×640px, boîte GT ≈ 32×32px)
  00016487_002.png  →  1 boîte(s)  (image 640×640px, boîte GT ≈ 32×32px)
  00012261_000.png  →  1 boîte(s)  (image 640×640px, boîte GT ≈ 32×32px)
  00018738_000.png  →  1 boîte(s)  (image 640×640px, boîte GT ≈ 32×32px)
  0264.png  →  2 boîte(s)  (image 640×640px, boîte GT ≈ 32×32px)
  00008534_000.png  →  1 boîte(s)  (image 640×640px, boîte GT ≈ 32×32px)
  00018738_005.png  →  1 boîte(s)  (image 640×640px, boîte GT ≈ 32×32px)
  0222.png  →  1 boîte(s)  (image 640×640px, boîte GT ≈ 32×32px)

  Annotations totales           : 153
  Images annotées uniques       : 97
  Centre X — min=137.75  max=1881.0  moy=958
  Centre Y — min=325.75  max=1649.0  moy=909

  Images de diagnostic sauvegardées dans 'diagnostic/'
  → Ouvre ces images pour vérifier que les croix rouges

/home/serveur/Documents/ecole/tf_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


WARNING ⚠️ cache='ram' may produce non-deterministic training results. Consider cache='disk' as a deterministic alternative if your disk space allows.
val: Caching images (0.2GB RAM): 100% ━━━━━━━━━━━━ 132/132 1.3Kit/s 0.1s
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.001' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
Plotting labels to /mnt/DATA/nodule_detection/runs/detect/nodule_finetuned_iter1/labels.jpg... 
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to /mnt/DATA/nodule_detection/runs/detect/nodule_finetuned_iter1
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/100      1.07G      3.292      7.985      2.184          6        640: 100% ━━━━━━━━━━━━ 48/48 12.2it/s 3.9s0.1s
                 Class    

...

Dataset YOLO créé dans 'yolo_dataset_preprocess_iter2'  (1891 train / 19 val)
YAML écrit : nodule_dataset.yaml
  Images train : 1933  |  Images val : 64
Ultralytics 8.4.21 🚀 Python-3.12.3 torch-2.10.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4070 SUPER, 11874MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=nodule_dataset.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.3, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=runs/detect/nodul

  Évaluation: 100%|████████████████████████████████████████████████████████████████████| 16/16 [00:03<00:00,  4.54img/s]



--- RÉSULTATS PIPELINE SUR IMAGES ANNOTÉES ---
  Images évaluées                    : 16
  Images franchissant le classifieur  : 16  (Sensibilité: 100.00%)

--- RÉSULTATS LOCALISATION ---
  GT nodules                          : 26
  Nodules prédits                     : 44
  TP (correctement localisés)         : 21
  FP (en trop)                        : 23
  FN (manqués)                        : 5
  Précision                           : 0.4773
  Rappel                              : 0.8077
  F1                                  : 0.6000
  Distance moyenne centre-à-centre    : 6.61 px

[VISU] Classification sur 500 malades + 500 saines


  Classification: 100%|████████████████████████████████████████████████████████████| 1000/1000 [00:30<00:00, 32.62img/s]



  Matrice de confusion (image-level) :
    TP=437  FP=296
    FN=63  TN=204
  Accuracy    : 0.641
  Precision   : 0.596
  Rappel      : 0.874  (sensibilite)
  Specificite : 0.408
  F1          : 0.709
  Grille sauvegardee -> visualisation_predictions.png

Pipeline terminé.


In [6]:
    metrics = evaluate_distance(
        df_bbox          = df_bbox_test,
        df_labels        = labels,
        classifier       = classifier,
        localisateur     = localisateur,
        seuil_conf_class = SEUIL_CONF_CLASS,
        seuil_conf_local = 0.2,
        seuil_rattrapage = 0.4,
    )

  Évaluation: 100%|████████████████████████████████████████████████████████████████████| 16/16 [00:03<00:00,  4.52img/s]


--- RÉSULTATS PIPELINE SUR IMAGES ANNOTÉES ---
  Images évaluées                    : 16
  Images franchissant le classifieur  : 16  (Sensibilité: 100.00%)

--- RÉSULTATS LOCALISATION ---
  GT nodules                          : 26
  Nodules prédits                     : 32
  TP (correctement localisés)         : 20
  FP (en trop)                        : 12
  FN (manqués)                        : 6
  Précision                           : 0.6250
  Rappel                              : 0.7692
  F1                                  : 0.6897
  Distance moyenne centre-à-centre    : 6.79 px


In [7]:
final_model_path

'runs/detect/nodule_finetuned_iter2/weights/best.pt'

In [14]:
visualize_predictions(
df_bbox          = df_bbox_test,
df_labels        = labels,
classifier       = classifier,
n_malades        = 500,
n_saines         = 500,
n_display        = 16,
seuil_conf_class = SEUIL_CONF_CLASS,
save_path        = "visualisation_predictions.png",
)


[VISU] Classification sur 500 malades + 500 saines


  Classification: 100%|████████████████████████████████████████████████████████████| 1000/1000 [00:30<00:00, 32.38img/s]



  Matrice de confusion (image-level) :
    TP=437  FP=296
    FN=63  TN=204
  Accuracy    : 0.641
  Precision   : 0.596
  Rappel      : 0.874  (sensibilite)
  Specificite : 0.408
  F1          : 0.709
  Grille sauvegardee -> visualisation_predictions.png


{'tp': 437,
 'fp': 296,
 'fn': 63,
 'tn': 204,
 'accuracy': 0.641,
 'precision': 0.5961800818553888,
 'recall': 0.874,
 'specificity': 0.408,
 'f1': 0.7088402270884023}

In [22]:
SEUIL_RATTRAPAGE=0.6

In [23]:
def visualize_predictions(
        df_bbox: pd.DataFrame,
        df_labels: pd.DataFrame,
        classifier: "Classifieur",
        localisateur ,
        n_malades:  int   = 500,
        n_saines:   int   = 500,
        n_display:  int   = 16,
        seuil_conf_class: float = SEUIL_CONF_CLASS,
        save_path: str = "visualisation_predictions.png",
) -> dict:
    """
    Evalue le CLASSIFIEUR sur n_malades images (label=1, tirées de df_labels)
    + n_saines images (label=0), indépendamment de df_bbox.

    Métriques : TP/FP/FN/TN au niveau image (classification binaire).
      TP = malade  prédit positif
      FP = saine   prédite positive
      FN = malade  prédit négatif
      TN = saine   prédite négative

    Affiche une grille de n_display images malades avec :
      + croix rouge  = position GT du nodule (si annotée dans df_bbox)
      Titre : Clf OK/NOK  |  résultat correct ou non
    """
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches

    random.seed(SEED)

    # 1. Echantillonnage depuis df_labels (pas seulement df_bbox)
    df_malades = df_labels[df_labels["label"] == 1]
    fnames_malades = df_malades["file_name"].tolist()
    random.shuffle(fnames_malades)
    fnames_malades = fnames_malades[:n_malades]

    df_saines = df_labels[df_labels["label"] == 0]
    fnames_saines = df_saines["file_name"].tolist()
    random.shuffle(fnames_saines)
    fnames_saines = fnames_saines[:n_saines]

    print(f"\n[VISU] Classification sur {len(fnames_malades)} malades + {len(fnames_saines)} saines")

    # 2. Inference classifieur uniquement
    tp = fp = fn = tn = 0
    display_data = []

    all_fnames = [(f, 1) for f in fnames_malades] + [(f, 0) for f in fnames_saines]

    for fname, gt_label in tqdm(all_fnames, desc="  Classification", unit="img"):
        row_lbl = df_labels[df_labels["file_name"] == fname]
        if row_lbl.empty:
            continue
        path_img = row_lbl.iloc[0]["path"]

        try:
            img_np = _load_image_from_disk(path_img)
        except Exception:
            continue

        pred_class = classifier.pred(img_np, threshold=seuil_conf_class).item()

        if not pred_class : pred_class = 1 if localisateur.pred(img_np, conf_threshold=SEUIL_RATTRAPAGE) else 0
        
        # Matrice de confusion image-level
        if gt_label == 1 and pred_class == 1:
            tp += 1
        elif gt_label == 0 and pred_class == 1:
            fp += 1
        elif gt_label == 1 and pred_class == 0:
            fn += 1
        else:
            tn += 1

        # GT nodules si disponibles (juste pour l'affichage)
        gt_pts = []
        rows_gt = df_bbox[df_bbox["file_name"] == fname]
        if not rows_gt.empty:
            gt_pts = [{"x": int(r["x"]), "y": int(r["y"])} for _, r in rows_gt.iterrows()]

        correct = (pred_class == gt_label)
        if gt_label == 1 and len(display_data) < n_display:
            display_data.append({
                "fname":      fname,
                "img_np":     img_np,
                "gt_pts":     gt_pts,
                "pred_class": pred_class,
                "correct":    correct,
            })

    # 3. Métriques classification
    total     = tp + fp + fn + tn
    accuracy  = (tp + tn) / total if total > 0 else 0
    precision = tp / (tp + fp)    if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn)    if (tp + fn) > 0 else 0
    f1        = 2*precision*recall / (precision+recall) if (precision+recall) > 0 else 0
    specificity = tn / (tn + fp)  if (tn + fp) > 0 else 0

    print(f"\n  Matrice de confusion (image-level) :")
    print(f"    TP={tp}  FP={fp}")
    print(f"    FN={fn}  TN={tn}")
    print(f"  Accuracy    : {accuracy:.3f}")
    print(f"  Precision   : {precision:.3f}")
    print(f"  Rappel      : {recall:.3f}  (sensibilite)")
    print(f"  Specificite : {specificity:.3f}")
    print(f"  F1          : {f1:.3f}")

    # 4. Grille matplotlib (images malades uniquement)
    n_cols = 4
    n_rows = math.ceil(len(display_data) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(n_cols * 4, n_rows * 4),
                             facecolor="#111111")
    axes = np.array(axes).flatten()

    for ax, data in zip(axes, display_data):
        img_rgb = preprocess_image(data["img_np"])
        img_lb, scale, pad_x, pad_y = _letterbox(img_rgb, YOLO_IMG_SIZE)
        ax.imshow(img_lb)
        ax.axis("off")

        # Croix rouge = GT nodule si annoté dans df_bbox
        for gt in data["gt_pts"]:
            cx_lb = gt["x"] * scale + pad_x
            cy_lb = gt["y"] * scale + pad_y
            s = 20
            ax.plot([cx_lb - s, cx_lb + s], [cy_lb,     cy_lb    ], color="#ff3333", lw=2.5)
            ax.plot([cx_lb,     cx_lb    ], [cy_lb - s, cy_lb + s], color="#ff3333", lw=2.5)

        # Titre : résultat classifieur
        clf_sym   = "POSITIF" if data["pred_class"] == 1 else "NEGATIF"
        title_col = "#aaffaa" if data["correct"] else "#ff6666"
        status    = "OK" if data["correct"] else "ERREUR"
        has_gt    = "annotee" if data["gt_pts"] else "non annotee"
        ax.set_title(
            data["fname"][:22] + "\n" + "Clf:{} [{}] {}".format(clf_sym, status, has_gt),
            fontsize=7, color=title_col, pad=3
        )

        # Bordure colorée selon correction
        for spine in ax.spines.values():
            spine.set_edgecolor("#aaffaa" if data["correct"] else "#ff6666")
            spine.set_linewidth(2)
            spine.set_visible(True)

    for ax in axes[len(display_data):]:
        ax.set_facecolor("#111111")
        ax.axis("off")

    legend_elems = [
        mpatches.Patch(color="#ff3333", label="GT nodule (croix)"),
        mpatches.Patch(color="#aaffaa", label="Classification correcte"),
        mpatches.Patch(color="#ff6666", label="Classification incorrecte"),
    ]
    fig.legend(handles=legend_elems, loc="lower center", ncol=3,
               fontsize=9, framealpha=0.85, bbox_to_anchor=(0.5, 0.0))

    suptitle = (
        "Classification  --  {} malades / {} saines  (seuil={:.2f})\n"
        "Acc={:.2f}  P={:.2f}  R={:.2f}  F1={:.2f}  Spec={:.2f}"
    ).format(len(fnames_malades), len(fnames_saines), seuil_conf_class,
             accuracy, precision, recall, f1, specificity)
    fig.suptitle(suptitle, fontsize=11, color="white", y=1.01)

    plt.tight_layout(rect=[0, 0.04, 1, 1])
    plt.savefig(save_path, dpi=130, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close()
    print(f"  Grille sauvegardee -> {save_path}")

    return {
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
        "accuracy": accuracy, "precision": precision,
        "recall": recall, "specificity": specificity, "f1": f1,
    }

In [24]:
visualize_predictions(
df_bbox          = df_bbox_test,
df_labels        = labels,
classifier       = classifier,
    localisateur=localisateur,
n_malades        = 500,
n_saines         = 500,
n_display        = 16,
seuil_conf_class = SEUIL_CONF_CLASS,
save_path        = "visualisation_predictions.png",
)


[VISU] Classification sur 500 malades + 500 saines


  Classification: 100%|████████████████████████████████████████████████████████████| 1000/1000 [00:37<00:00, 26.61img/s]



  Matrice de confusion (image-level) :
    TP=462  FP=398
    FN=38  TN=102
  Accuracy    : 0.564
  Precision   : 0.537
  Rappel      : 0.924  (sensibilite)
  Specificite : 0.204
  F1          : 0.679
  Grille sauvegardee -> visualisation_predictions.png


{'tp': 462,
 'fp': 398,
 'fn': 38,
 'tn': 102,
 'accuracy': 0.564,
 'precision': 0.5372093023255814,
 'recall': 0.924,
 'specificity': 0.204,
 'f1': 0.6794117647058824}

In [28]:
def grid_search_thresholds(
        df_labels: pd.DataFrame,
        classifier: "Classifieur",
        localisateur: "Localisation",
        n_malades: int = 500,
        n_saines: int = 500,
        save_path: str = "grid_search_heatmap.png"
) -> Tuple[float, float, pd.DataFrame]:
    """
    Explore les combinaisons de (seuil_conf_class, seuil_rattrapage)
    pour maximiser le F1-Score global sur un échantillon d'images dézippées.
    """
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import seaborn as sns
    import torch
    import torch.nn.functional as F
    
    print("\n[GRID SEARCH] Échantillonnage des données...")
    random.seed(SEED)
    
    # 1. Echantillonnage
    df_malades = df_labels[df_labels["label"] == 1]
    df_malades = df_malades.sample(n=min(n_malades, len(df_malades)), random_state=SEED)
    
    df_saines = df_labels[df_labels["label"] == 0]
    df_saines = df_saines.sample(n=min(n_saines, len(df_saines)), random_state=SEED)
    
    df_eval = pd.concat([df_malades, df_saines]).sample(frac=1, random_state=SEED)
    
    # 2. Pré-calcul des scores bruts (1 seule inférence par image)
    eval_data = []
    
    for _, row in tqdm(df_eval.iterrows(), total=len(df_eval), desc="  Pré-calcul des inférences", unit="img"):
        path_img = row["path"]
        gt_label = row["label"]
        
        try:
            img_np = _load_image_from_disk(path_img)
        except Exception:
            continue
            
        # A. Score brut du classifieur (probabilité entre 0.0 et 1.0)
        img_tensor = classifier.preprocess(img_np)
        with torch.no_grad():
            outputs = classifier.model(img_tensor)
            probs = F.softmax(outputs, dim=1)
            score_clf = probs[0, 1].item()
            
        # B. Confiance max de YOLO (on prend un seuil très bas pour tout lire)
        nodules = localisateur.pred(img_np, conf_threshold=0.01)
        max_yolo = max([n['conf'] for n in nodules]) if nodules else 0.0
        
        eval_data.append({
            "gt": gt_label,
            "score_clf": score_clf,
            "max_yolo": max_yolo
        })
        
    df_res = pd.DataFrame(eval_data)
    
    # 3. Grid Search vectorisé (instantané)
    seuils_clf = np.round(np.arange(0.10, 0.94, 0.02), 2)
    seuils_rat = np.round(np.arange(0.10, 0.94, 0.02), 2)
    
    results = []
    best_f1 = 0
    best_params = (0.5, 0.5)
    
    for s_clf in tqdm(seuils_clf, desc="  Exploration de la grille", unit="seuil"):
        for s_rat in seuils_rat:
            # Logique du pipeline : Positif si le classifieur dit OUI (>= s_clf),
            # OU (rattrapage) si YOLO dit OUI (>= s_rat).
            preds = ((df_res["score_clf"] >= s_clf) | (df_res["max_yolo"] >= s_rat)).astype(int)
            
            tp = ((preds == 1) & (df_res["gt"] == 1)).sum()
            fp = ((preds == 1) & (df_res["gt"] == 0)).sum()
            fn = ((preds == 0) & (df_res["gt"] == 1)).sum()
            tn = ((preds == 0) & (df_res["gt"] == 0)).sum()
            
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
            
            results.append({
                "seuil_clf": s_clf,
                "seuil_rat": s_rat,
                "f1": f1,
                "precision": precision,
                "recall": recall
            })
            
            if f1 > best_f1:
                best_f1 = f1
                best_params = (s_clf, s_rat)
                
    df_grid = pd.DataFrame(results)
    
    # 4. Affichage Heatmap
    pivot_f1 = df_grid.pivot(index="seuil_clf", columns="seuil_rat", values="f1")
    
    plt.figure(figsize=(10, 8), facecolor="#111111")
    ax = plt.gca()
    ax.set_facecolor("#111111")
    
    sns.heatmap(pivot_f1, annot=True, fmt=".3f", cmap="viridis", cbar_kws={"label": "F1-Score"})
    
    plt.title("Grid-Search : Optimisation du F1-Score", color="white", pad=15, fontsize=14)
    plt.xlabel("Seuil de Rattrapage YOLO", color="white", fontsize=12)
    plt.ylabel("Seuil du Classifieur", color="white", fontsize=12)
    plt.xticks(color="white")
    plt.yticks(color="white")
    
    # Couleur de la colorbar
    cbar = ax.collections[0].colorbar
    cbar.ax.yaxis.set_tick_params(color="white")
    plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color="white")
    cbar.set_label("F1-Score", color="white")

    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(save_path, dpi=130, facecolor=plt.gcf().get_facecolor())
    plt.close()
    
    print("\n--- RÉSULTATS GRID SEARCH ---")
    print(f"Meilleur Seuil Classifieur : {best_params[0]:.2f}")
    print(f"Meilleur Seuil Rattrapage  : {best_params[1]:.2f}")
    print(f"F1-Score Max               : {best_f1:.4f}")
    print(f"Graphique sauvegardé       : {save_path}")
    
    return best_params[0], best_params[1], df_grid

In [29]:
# --- Optimisation des Seuils ---
best_seuil_clf, best_seuil_rat, df_grid = grid_search_thresholds(
    df_labels=labels,
    classifier=classifier,
    localisateur=localisateur,
    n_malades=500,
    n_saines=500,
    save_path="grid_search_heatmap.png"
)




[GRID SEARCH] Échantillonnage des données...


  Exploration de la grille: 100%|████████████████████████████████████████████████████| 42/42 [00:00<00:00, 69.29seuil/s]



--- RÉSULTATS GRID SEARCH ---
Meilleur Seuil Classifieur : 0.38
Meilleur Seuil Rattrapage  : 0.86
F1-Score Max               : 0.6976
Graphique sauvegardé       : grid_search_heatmap.png
